# Insurance Notebook: Fraud Triage Optimization (Expanded)

This expanded notebook builds a realistic insurance claims workflow for fraud triage under operational constraints.

What is included:
- richer synthetic claims simulation,
- exploratory fraud diagnostics,
- model benchmarking (logistic vs random forest),
- calibration and score-banding,
- capacity-aware queue simulation,
- investigation economics and policy comparison,
- fairness-style monitoring across channels/regions.

## 0) Imports and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

np.random.seed(7)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1) Simulate a richer claims portfolio

In [ ]:
n = 18000

claims = pd.DataFrame(
    {
        "claim_amount": np.random.lognormal(mean=8.3, sigma=0.72, size=n),
        "days_to_report": np.random.gamma(shape=2.2, scale=7.0, size=n),
        "num_prior_claims": np.random.poisson(lam=1.2, size=n),
        "policy_tenure_months": np.random.randint(1, 220, size=n),
        "driver_age": np.random.randint(18, 85, size=n),
        "channel": np.random.choice(["agent", "online", "phone", "broker"], p=[0.42, 0.30, 0.16, 0.12], size=n),
        "claim_type": np.random.choice(["auto", "property", "injury", "theft"], p=[0.50, 0.25, 0.15, 0.10], size=n),
        "region": np.random.choice(["North", "South", "East", "West"], p=[0.25, 0.30, 0.22, 0.23], size=n),
    }
)

claims["weekend_report"] = np.random.binomial(1, 0.24, size=n)
claims["night_report"] = np.random.binomial(1, 0.18, size=n)

# Latent fraud propensity
signal = (
    0.00009 * claims["claim_amount"].to_numpy()
    + 0.030 * np.log1p(claims["days_to_report"].to_numpy())
    + 0.24 * np.log1p(claims["num_prior_claims"].to_numpy())
    - 0.0038 * claims["policy_tenure_months"].to_numpy()
    + 0.11 * claims["weekend_report"].to_numpy()
    + 0.10 * claims["night_report"].to_numpy()
    + np.where(claims["channel"].to_numpy() == "online", 0.27, 0.0)
    + np.where(claims["claim_type"].to_numpy() == "injury", 0.34, 0.0)
    + np.where(claims["claim_type"].to_numpy() == "theft", 0.21, 0.0)
    + np.where(claims["region"].to_numpy() == "South", 0.13, 0.0)
    + np.where(claims["driver_age"].to_numpy() < 23, 0.10, 0.0)
)

p_fraud = 1 / (1 + np.exp(-(signal - 1.25)))
claims["is_fraud"] = np.random.binomial(1, p_fraud, size=n)

# Investigation/recovery assumptions
claims["recoverable_amount"] = claims["claim_amount"] * np.random.uniform(0.45, 0.8, size=n)

claims.head()

In [ ]:
print(f"Claims: {len(claims):,}")
print(f"Fraud prevalence: {claims['is_fraud'].mean():.2%}")
print(f"Avg claim amount: {claims['claim_amount'].mean():.0f}")

## 2) Exploratory fraud diagnostics

In [ ]:
fraud_by_type = (
    claims.groupby("claim_type", as_index=False)
    .agg(fraud_rate=("is_fraud", "mean"), n_claims=("is_fraud", "size"), avg_amount=("claim_amount", "mean"))
    .sort_values("fraud_rate", ascending=False)
)

fraud_by_channel = (
    claims.groupby("channel", as_index=False)
    .agg(fraud_rate=("is_fraud", "mean"), n_claims=("is_fraud", "size"))
    .sort_values("fraud_rate", ascending=False)
)

fraud_by_type, fraud_by_channel

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

sns.barplot(data=fraud_by_type, x="claim_type", y="fraud_rate", ax=ax[0], palette="Reds")
ax[0].set_title("Fraud Rate by Claim Type")
ax[0].set_ylabel("Fraud rate")

sns.barplot(data=fraud_by_channel, x="channel", y="fraud_rate", ax=ax[1], palette="Blues")
ax[1].set_title("Fraud Rate by Intake Channel")
ax[1].set_ylabel("Fraud rate")

plt.tight_layout()
plt.show()

## 3) Data split and preprocessing

In [ ]:
features = [
    "claim_amount",
    "days_to_report",
    "num_prior_claims",
    "policy_tenure_months",
    "driver_age",
    "channel",
    "claim_type",
    "region",
    "weekend_report",
    "night_report",
]

target = "is_fraud"

X = claims[features]
y = claims[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7, stratify=y
)

num_cols = [
    "claim_amount",
    "days_to_report",
    "num_prior_claims",
    "policy_tenure_months",
    "driver_age",
    "weekend_report",
    "night_report",
]
cat_cols = ["channel", "claim_type", "region"]

prep = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

## 4) Benchmark models

In [ ]:
logit = Pipeline(
    steps=[
        ("prep", prep),
        ("clf", LogisticRegression(max_iter=500, random_state=7)),
    ]
)

rf = Pipeline(
    steps=[
        ("prep", prep),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=350,
                max_depth=10,
                min_samples_leaf=10,
                random_state=7,
                n_jobs=-1,
            ),
        ),
    ]
)

logit.fit(X_train, y_train)
rf.fit(X_train, y_train)

preds = {
    "Logistic": logit.predict_proba(X_test)[:, 1],
    "RandomForest": rf.predict_proba(X_test)[:, 1],
}

rows = []
for name, p in preds.items():
    rows.append(
        {
            "model": name,
            "roc_auc": roc_auc_score(y_test, p),
            "pr_auc": average_precision_score(y_test, p),
            "brier": brier_score_loss(y_test, p),
        }
    )

metrics = pd.DataFrame(rows).sort_values("pr_auc", ascending=False)
metrics

In [ ]:
plt.figure(figsize=(7, 4))
plot_df = metrics.melt(id_vars="model", value_vars=["roc_auc", "pr_auc", "brier"], var_name="metric")
sns.barplot(data=plot_df, x="metric", y="value", hue="model", palette="Set2")
plt.title("Fraud Model Benchmark")
plt.tight_layout()
plt.show()

## 5) Champion model calibration and score bands

In [ ]:
champion_name = metrics.iloc[0]["model"]
champion = logit if champion_name == "Logistic" else rf
proba_test = preds[champion_name]

print(f"Champion model: {champion_name}")
print(f"ROC-AUC={roc_auc_score(y_test, proba_test):.3f} | PR-AUC={average_precision_score(y_test, proba_test):.3f}")

frac_pos, mean_pred = calibration_curve(y_test, proba_test, n_bins=12)

plt.figure(figsize=(6, 5.2))
plt.plot(mean_pred, frac_pos, marker="o", label="Champion")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect")
plt.xlabel("Mean predicted fraud probability")
plt.ylabel("Observed fraud rate")
plt.title("Calibration Curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
score_band = pd.DataFrame({"score": proba_test, "label": y_test.values})
score_band["band"] = pd.qcut(score_band["score"], q=10, labels=False, duplicates="drop")
band_summary = (
    score_band.groupby("band", as_index=False)
    .agg(avg_score=("score", "mean"), obs_fraud_rate=("label", "mean"), n=("label", "size"))
    .sort_values("band")
)
band_summary

## 6) Capacity-constrained triage simulation

In [ ]:
test_df = X_test.copy()
test_df["label"] = y_test.values
test_df["score"] = proba_test
test_df["recoverable_amount"] = claims.loc[test_df.index, "recoverable_amount"]

def evaluate_policy(df: pd.DataFrame, capacity_ratio: float, investigation_cost: float = 180.0, recovery_rate: float = 0.62):
    k = int(len(df) * capacity_ratio)
    ranked = df.sort_values("score", ascending=False).head(k)

    tp = int((ranked["label"] == 1).sum())
    fp = int((ranked["label"] == 0).sum())
    total_fraud = int(df["label"].sum())

    precision = tp / max(k, 1)
    recall = tp / max(total_fraud, 1)

    expected_recovery = (ranked.loc[ranked["label"] == 1, "recoverable_amount"].sum()) * recovery_rate
    investigation_spend = k * investigation_cost
    net_benefit = expected_recovery - investigation_spend

    return {
        "capacity_ratio": capacity_ratio,
        "n_reviewed": k,
        "tp_detected": tp,
        "fp_reviewed": fp,
        "precision": precision,
        "recall": recall,
        "expected_recovery": expected_recovery,
        "investigation_spend": investigation_spend,
        "net_benefit": net_benefit,
    }

capacity_grid = [0.02, 0.03, 0.05, 0.08, 0.10, 0.12, 0.15]
policy_table = pd.DataFrame([evaluate_policy(test_df, c) for c in capacity_grid])
policy_table

In [ ]:
best_policy = policy_table.sort_values("net_benefit", ascending=False).iloc[0]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

ax[0].plot(policy_table["capacity_ratio"], policy_table["net_benefit"], marker="o", color="#55A868")
ax[0].axvline(best_policy["capacity_ratio"], linestyle="--", color="black")
ax[0].set_title("Capacity vs Net Benefit")
ax[0].set_xlabel("Investigation capacity ratio")
ax[0].set_ylabel("Net benefit")

ax[1].plot(policy_table["capacity_ratio"], policy_table["precision"], marker="o", label="Precision", color="#4C72B0")
ax[1].plot(policy_table["capacity_ratio"], policy_table["recall"], marker="o", label="Recall", color="#C44E52")
ax[1].axvline(best_policy["capacity_ratio"], linestyle="--", color="black")
ax[1].set_title("Operating Metrics vs Capacity")
ax[1].set_xlabel("Investigation capacity ratio")
ax[1].legend()

plt.tight_layout()
plt.show()

print(
    f"Best capacity ratio: {best_policy['capacity_ratio']:.0%} | "
    f"Reviewed={int(best_policy['n_reviewed'])} | "
    f"Net benefit={best_policy['net_benefit']:.0f}"
)

## 7) Score threshold policy view

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, proba_test)

th_rows = []
for t in np.linspace(0.10, 0.90, 60):
    pred = (proba_test >= t).astype(int)
    tp = int(((pred == 1) & (y_test.values == 1)).sum())
    fp = int(((pred == 1) & (y_test.values == 0)).sum())
    fn = int(((pred == 0) & (y_test.values == 1)).sum())

    th_rows.append(
        {
            "threshold": t,
            "review_rate": pred.mean(),
            "precision": tp / max(tp + fp, 1),
            "recall": tp / max(tp + fn, 1),
        }
    )

th_table = pd.DataFrame(th_rows)

plt.figure(figsize=(8, 4.5))
plt.plot(th_table["threshold"], th_table["review_rate"], label="Review rate", color="#8172B2")
plt.plot(th_table["threshold"], th_table["precision"], label="Precision", color="#4C72B0")
plt.plot(th_table["threshold"], th_table["recall"], label="Recall", color="#C44E52")
plt.xlabel("Score threshold")
plt.title("Threshold Policy Curves")
plt.legend()
plt.tight_layout()
plt.show()

## 8) Monitoring slices (channel and region)

In [ ]:
# Evaluate uplift from model ranking in each operational slice
selected_ratio = float(best_policy["capacity_ratio"])

slice_rows = []
for dim in ["channel", "region"]:
    for value, grp in test_df.groupby(dim):
        k = int(len(grp) * selected_ratio)
        if k < 1:
            continue
        ranked = grp.sort_values("score", ascending=False).head(k)

        slice_rows.append(
            {
                "dimension": dim,
                "slice": value,
                "n": len(grp),
                "base_fraud_rate": grp["label"].mean(),
                "precision_at_capacity": ranked["label"].mean(),
                "lift": ranked["label"].mean() / max(grp["label"].mean(), 1e-9),
            }
        )

slice_perf = pd.DataFrame(slice_rows).sort_values(["dimension", "lift"], ascending=[True, False])
slice_perf

In [ ]:
plt.figure(figsize=(9, 4.8))
plot_df = slice_perf[slice_perf["dimension"] == "channel"].copy()
sns.barplot(data=plot_df, x="slice", y="lift", palette="mako")
plt.axhline(1.0, linestyle="--", color="black")
plt.title("Channel-Level Precision Lift at Selected Capacity")
plt.ylabel("Lift vs base fraud rate")
plt.xlabel("Channel")
plt.tight_layout()
plt.show()

## 9) Final summary

- The workflow now supports model governance, not just classification metrics.
- Capacity-aware and economics-aware policies identify realistic operating points for SIU teams.
- Slice-level monitoring can be used as an early warning for uneven model behavior across channels and regions.